# Step 3 — Feature engineering

Loads `data/matches_clean.parquet` (from `explore.ipynb`) and builds per-match features,
every one knowable **before kickoff**. Writes `data/model_df.parquet`.

- **3.1** Reshape to one row per team per match
- **3.2** Rolling form + **EWMA** (continuous decay, no 5-game cliff) — the leak-free core
- **3.3** Venue split, rest days, congestion, head-to-head
- **3.4** Advanced:
  - **a** rolling shot quality, **a-bis** an **expected-goals proxy** (npxG-style, fit by OLS)
  - **b** **margin-aware Elo** with a moving home-field advantage (strength of schedule)
  - **c** opponent-weighted form, derby flag
  - **d** pre-match league position & relegation pressure
  - **e** **market signal** — vig-free (proportional + Shin) 1X2 probabilities, implied
    total goals from the Over/Under line, closing-line movement
- **3.5** Merge team features back as `home_*` / `away_*` / `*_diff`
- **3.6** Leakage checks + `train_ready` view
- **3.7** Squad-strength module from the **FPL Core Insights** player data

Running example throughout the display cells: **Man United**.

The rule: a feature for match *k* may only use matches *1…k-1*. Every rolling stat is
`shift(1)` before it is aggregated; Elo is recorded pre-update; odds are the pre-kickoff
price; FPL squad prices are the pre-season snapshot. `home_goals` / `away_goals` are copied
in only as *targets* for the goal model — never read back as inputs.


In [ ]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

DATA_DIR = "data"
matches = pd.read_parquet(os.path.join(DATA_DIR, "matches_clean.parquet"))
matches = matches.sort_values("Date").reset_index(drop=True)
matches["match_id"] = matches.index
print(matches.shape)
matches.head()

## 3.1 Reshape to one row per team per match

Each match -> two rows (home team's view, away team's view). We record what the team
*did* in that match (goals for/against, shots, points). These are match **outcomes** —
3.2 rolls them forward with `shift(1)` so a row never sees its own match.

In [ ]:
def points(gf, ga):
    """League points from goals for / against."""
    return pd.Series(1, index=gf.index).mask(gf > ga, 3).mask(gf < ga, 0)


_common = dict(match_id=matches.match_id, Season=matches.Season, Date=matches.Date)
_home = pd.DataFrame({**_common,
    "team": matches.HomeTeam, "opponent": matches.AwayTeam, "venue": "home",
    "gf": matches.FTHG, "ga": matches.FTAG,
    "shots": matches.HS, "sot": matches.HST, "corners": matches.HC})
_away = pd.DataFrame({**_common,
    "team": matches.AwayTeam, "opponent": matches.HomeTeam, "venue": "away",
    "gf": matches.FTAG, "ga": matches.FTHG,
    "shots": matches.AS, "sot": matches.AST, "corners": matches.AC})

tm = pd.concat([_home, _away], ignore_index=True)
tm["pts"] = points(tm.gf, tm.ga)
tm["gd"] = tm.gf - tm.ga                       # goal difference this match
tm = tm.sort_values(["Date", "match_id"]).reset_index(drop=True)
print(tm.shape, " (expect", len(matches) * 2, "rows)")
tm.head(4)


## 3.2 Rolling form — the leak-free core

Per team, sorted by date, we want the mean of the team's **previous** N matches:

```
rolling(5).mean() at match k  -> includes match k        (leak)
shift(1) first, then roll     -> mean of matches k-5..k-1  (safe)
```

`_key(...)` collapses multi-column group keys into one Series so
`groupby(key).rolling(...)` returns a clean single-level result to re-align.

Two horizons: **last 5** (recent form) and **season-to-date** (`expanding`, resets each
season). `min_periods=1` -> a value after one prior game; a team's very first ever match
is `NaN`.

In [ ]:
ROLL_STATS = ["pts", "gf", "ga", "shots", "sot", "corners", "gd"]
WINDOW = 5
EWM_HALFLIFE = 4          # continuous decay — no hard 5-game cliff

tm = tm.sort_values(["team", "Date", "match_id"]).reset_index(drop=True)


def _key(*cols):
    """Collapse one or more columns into a single grouping-key Series."""
    if len(cols) == 1:
        return cols[0]
    out = cols[0].astype(str)
    for c in cols[1:]:
        out = out + "||" + c.astype(str)
    return out


def prior_rolling(s, key, window, how="mean"):
    """Aggregate the last `window` PRIOR rows within each group (shift, then roll)."""
    shifted = s.groupby(key).shift(1)
    rolled = shifted.groupby(key).rolling(window, min_periods=1)
    rolled = getattr(rolled, how)().reset_index(level=0, drop=True)
    return rolled.reindex(s.index)


def prior_expanding(s, key, how="mean"):
    """Aggregate ALL prior rows within each group (shift, then expand)."""
    shifted = s.groupby(key).shift(1)
    exp = shifted.groupby(key).expanding(min_periods=1)
    exp = getattr(exp, how)().reset_index(level=0, drop=True)
    return exp.reindex(s.index)


def prior_ewm(s, key, halflife):
    """Exponentially-weighted mean of PRIOR rows (shift, then ewm).

    Recent matches decay smoothly instead of a fixed window dropping match k-5
    out of the average in one step.
    """
    shifted = s.groupby(key).shift(1)
    ewm = shifted.groupby(key).ewm(halflife=halflife).mean()
    return ewm.reset_index(level=0, drop=True).reindex(s.index)


for stat in ROLL_STATS:
    tm[f"{stat}_roll{WINDOW}"] = prior_rolling(tm[stat], _key(tm.team), WINDOW)
    tm[f"{stat}_std"] = prior_expanding(tm[stat], _key(tm.team, tm.Season))
    tm[f"{stat}_ewm"] = prior_ewm(tm[stat], _key(tm.team), EWM_HALFLIFE)

# games already played this season (0 for the opener)
tm["games_played"] = tm.groupby(["team", "Season"]).cumcount()

# running example: Man United's opening 2015-16 matches — roll5 vs ewm side by side
EXAMPLE_TEAM = "Man United"
(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|opponent|Date|pts$|pts_roll5|pts_ewm|gd_roll5|gd_ewm|games_played")
   .head(8))


## 3.3 Venue split, rest days, head-to-head

- **venue form** — rolling points, restricted to this venue (home form vs away form)
- **days_rest** — days since this team's previous match; **congestion** — matches in the
  last 14 days
- **head-to-head** — this team's mean points in its last 5 meetings with this specific
  opponent (either venue), keyed on an unordered pair

In [ ]:
# venue-specific points form
tm[f"venue_pts_roll{WINDOW}"] = prior_rolling(tm.pts, _key(tm.team, tm.venue), WINDOW)

# rest & congestion
tm = tm.sort_values(["team", "Date", "match_id"]).reset_index(drop=True)
tm["days_rest"] = tm.groupby("team")["Date"].diff().dt.days


def matches_last_n_days(dates, n):
    """For each row, count this team's PRIOR matches within the last n days."""
    d = dates.values.astype("datetime64[D]")
    out = np.zeros(len(d), dtype=int)
    j = 0
    for i in range(len(d)):
        while d[i] - d[j] > np.timedelta64(n, "D"):
            j += 1
        out[i] = i - j            # prior matches inside the window
    return out


tm["congestion_14d"] = (
    tm.groupby("team", group_keys=False)["Date"]
      .apply(lambda s: pd.Series(matches_last_n_days(s.sort_values(), 14),
                                 index=s.sort_values().index))
      .reindex(tm.index)
)

# head-to-head (unordered pair key)
lo = tm[["team", "opponent"]].min(axis=1)
hi = tm[["team", "opponent"]].max(axis=1)
tm["pair"] = lo + " v " + hi
tm["h2h_pts_roll5"] = prior_rolling(tm.pts, _key(tm.team, tm.pair), 5)

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|opponent|Date|venue_pts|days_rest|congestion|h2h")
   .head(8))


## 3.4 Advanced features

### 3.4a Rolling shot quality / efficiency

Rolling **shot conversion** (goals / shots on target) and **finishing vs chances**
(goals − a crude 0.3·SoT expected). These regress to the mean, so a team on a hot/cold
streak is flagged. All `shift(1)`-based.

In [ ]:
# rolling sums so ratios use consistent denominators
for stat in ["gf", "ga", "shots", "sot"]:
    tm[f"{stat}_sum{WINDOW}"] = prior_rolling(tm[stat], _key(tm.team), WINDOW, how="sum")

tm["conv_roll5"] = tm.gf_sum5 / tm.sot_sum5.replace(0, np.nan)          # goals per SoT
tm["sot_rate_roll5"] = tm.sot_sum5 / tm.shots_sum5.replace(0, np.nan)   # shot accuracy

# true "SoT faced" = opponent's SoT in each of this team's matches, then rolled
opp_sot = (tm[["match_id", "team", "sot"]]
           .merge(tm[["match_id", "team", "sot"]], on="match_id", suffixes=("", "_opp")))
opp_sot = opp_sot[opp_sot.team != opp_sot.team_opp][["match_id", "team", "sot_opp"]]
tm = tm.merge(opp_sot, on=["match_id", "team"], how="left")
tm["sot_faced_sum5"] = prior_rolling(tm.sot_opp, _key(tm.team), WINDOW, how="sum")
tm["save_pct_roll5"] = 1 - tm.ga_sum5 / tm.sot_faced_sum5.replace(0, np.nan)

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Date|conv_roll5|sot_rate_roll5|save_pct_roll5")
   .head(8))


### 3.4a-bis  Expected-goals proxy (npxG-style)

Raw shot counts are noisy: a team can take 20 low-quality shots and score once. **Expected
goals** weights each shot by its chance of scoring. We don't have shot coordinates for 25
seasons, but we can build a solid proxy from the two shot columns we *do* have:

$$\widehat{xG} = a + b\cdot(\text{shots} - \text{SoT}) + c\cdot\text{SoT}$$

Coefficients `a, b, c` are fit by ordinary least squares of **actual goals** on
`(off-target shots, on-target shots)` — solved from the 3×3 normal equations, fit **only on
seasons ≤ 2018-19** so nothing downstream of the split leaks in. On-target shots carry
almost all the weight (`c ≈ 0.3`, i.e. ~1 in 3 shots on target is a goal), off-target shots
almost none — exactly the shape a real xG model has.

For **2026-27** the source CSV ships real `HxG / AxG`; we use those directly and fall back
to the proxy everywhere else. Rolling xG-for / xG-against then behave like the other form
features, plus `finishing_roll5 = goals − xG` flags a team riding hot or cold finishing
(it regresses to zero).


In [ ]:
# opponent shots onto each team-match row (we already have sot_opp; add shots_opp)
opp_shots = (tm[["match_id", "team", "shots"]]
             .merge(tm[["match_id", "team", "shots"]], on="match_id", suffixes=("", "_o")))
opp_shots = opp_shots[opp_shots.team != opp_shots.team_o][["match_id", "team", "shots_o"]]
tm = tm.merge(opp_shots.rename(columns={"shots_o": "shots_opp"}), on=["match_id", "team"], how="left")


def _det3(A):
    return (A[0, 0] * (A[1, 1] * A[2, 2] - A[1, 2] * A[2, 1])
            - A[0, 1] * (A[1, 0] * A[2, 2] - A[1, 2] * A[2, 0])
            + A[0, 2] * (A[1, 0] * A[2, 1] - A[1, 1] * A[2, 0]))


def ols3(y, x1, x2):
    """Least-squares y ~ a + b*x1 + c*x2 via the 3x3 normal equations, solved by
    Cramer's rule (keeps us off numpy.linalg, which Smart App Control blocks)."""
    m = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
    y, x1, x2 = y[m], x1[m], x2[m]
    n = float(len(y))
    S = np.array([
        [n,          x1.sum(),        x2.sum()],
        [x1.sum(),   (x1 * x1).sum(), (x1 * x2).sum()],
        [x2.sum(),   (x1 * x2).sum(), (x2 * x2).sum()],
    ])
    rhs = np.array([y.sum(), (x1 * y).sum(), (x2 * y).sum()])
    d0 = _det3(S)
    out = []
    for i in range(3):
        Ai = S.copy()
        Ai[:, i] = rhs
        out.append(_det3(Ai) / d0)
    return out   # a, b, c


FIT_SEASONS = [s for s in sorted(tm.Season.unique()) if s <= "2018-19"]
fit = tm[tm.Season.isin(FIT_SEASONS)]
a, b, c = ols3(fit.gf.to_numpy(float),
               (fit.shots - fit.sot).to_numpy(float),
               fit.sot.to_numpy(float))
print(f"xG proxy fit on {FIT_SEASONS[0]}..{FIT_SEASONS[-1]}:  "
      f"xG = {a:.3f} + {b:.4f}*(shots-SoT) + {c:.4f}*SoT")


def xg_proxy(shots, sot):
    return np.clip(a + b * (shots - sot) + c * sot, 0.0, None)


tm["xg"] = xg_proxy(tm.shots, tm.sot)
tm["xga"] = xg_proxy(tm.shots_opp, tm.sot_opp)

# real xG where the source provides it (2026-27): map match xG onto team rows
if "HxG" in matches.columns and matches["HxG"].notna().any():
    hx = matches.set_index("match_id")[["HxG", "AxG"]]
    for_map = tm["match_id"].map(hx["HxG"]).where(tm.venue == "home",
                                                  tm["match_id"].map(hx["AxG"]))
    against_map = tm["match_id"].map(hx["AxG"]).where(tm.venue == "home",
                                                     tm["match_id"].map(hx["HxG"]))
    tm["xg"] = for_map.where(for_map.notna(), tm["xg"])
    tm["xga"] = against_map.where(against_map.notna(), tm["xga"])
    print(f"real xG used for {int(for_map.notna().sum())} team-match rows (2026-27)")

# rolling / ewm xG form, and finishing over-performance
for stat in ["xg", "xga"]:
    tm[f"{stat}_roll{WINDOW}"] = prior_rolling(tm[stat], _key(tm.team), WINDOW)
    tm[f"{stat}_ewm"] = prior_ewm(tm[stat], _key(tm.team), EWM_HALFLIFE)

tm["xg_sum5"] = prior_rolling(tm.xg, _key(tm.team), WINDOW, how="sum")
tm["xga_sum5"] = prior_rolling(tm.xga, _key(tm.team), WINDOW, how="sum")
tm["xgd_roll5"] = tm.xg_roll5 - tm.xga_roll5
tm["xgd_ewm"] = tm.xg_ewm - tm.xga_ewm
tm["finishing_roll5"] = (tm.gf_sum5 - tm.xg_sum5) / WINDOW       # + = finishing above xG
tm["def_luck_roll5"] = (tm.xga_sum5 - tm.ga_sum5) / WINDOW       # + = conceding below xGA

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Date|xg_roll5|xga_roll5|xgd_roll5|finishing_roll5")
   .head(8))


### 3.4b Elo rating — margin-aware, with a moving home-field advantage

One number per team that updates after every match and **absorbs strength of schedule**.
Chess Elo, adapted for football with three upgrades over the textbook version:

- **expected score** `E = 1 / (1 + 10**(-(R_home + HFA - R_away)/400))`
- **update** `R += K_eff * (S - E)`, `S ∈ {1, 0.5, 0}` for W / D / L
- **margin-of-victory scaling** (World Football Elo): a 5–0 must move ratings more than a
  1–0. `K_eff = K * sqrt(max(|Δgoals|, 1))`, so a 3-goal win updates ~1.7× a 1-goal win.
- **dynamic home-field advantage**: home edge has drifted down over 25 seasons and
  collapsed in the empty-stadium 2020-21 season. Instead of a fixed `HFA = 60`, we feed
  each season the **home advantage implied by the previous 3 seasons' results**
  (`HFA = -400·log10(1/S̄_home − 1)`, `S̄_home` = league mean home points-rate), so the
  model isn't told to expect a 2005-size home boost in 2021.
- ratings carry across seasons but **regress 25% toward 1500** each season start;
  promoted teams enter at 1500.

Stored per match: both teams' pre-match Elo, their difference, the model's implied
home-win-equivalent `elo_exp_home`, and the `hfa_used` that season.


In [ ]:
K = 20
BASE = 1500.0
REGRESS = 0.25   # toward BASE at each season boundary
DEFAULT_HFA = 60.0

# --- dynamic home-field advantage: previous-3-seasons implied home edge ----------
season_home_rate = (matches.assign(hp=matches.FTR.map({"H": 1.0, "D": 0.5, "A": 0.0}))
                    .groupby("Season").hp.mean())


def hfa_from_rate(sbar):
    sbar = min(max(sbar, 0.5001), 0.75)          # keep the log finite / sane
    return -400.0 * np.log10(1.0 / sbar - 1.0)


seasons_sorted = sorted(matches.Season.unique())
HFA_BY_SEASON = {}
for i, s in enumerate(seasons_sorted):
    prev = seasons_sorted[max(0, i - 3):i]
    if prev:
        HFA_BY_SEASON[s] = hfa_from_rate(season_home_rate.loc[prev].mean())
    else:
        HFA_BY_SEASON[s] = DEFAULT_HFA

print("home-field advantage fed to Elo (rating pts), by season:")
print(pd.Series(HFA_BY_SEASON).round(1).to_string())

# --- the Elo loop ---------------------------------------------------------------
elo, seen_season = {}, {}
home_elo_pre, away_elo_pre, exp_home, hfa_used = [], [], [], []

for row in matches.itertuples(index=False):
    hfa = HFA_BY_SEASON[row.Season]
    for t in (row.HomeTeam, row.AwayTeam):
        if t not in elo:
            elo[t] = BASE
        if seen_season.get(t) != row.Season:            # season rollover for this team
            elo[t] = BASE + (1 - REGRESS) * (elo[t] - BASE)
            seen_season[t] = row.Season

    rh, ra = elo[row.HomeTeam], elo[row.AwayTeam]
    e_h = 1 / (1 + 10 ** (-((rh + hfa) - ra) / 400))
    home_elo_pre.append(rh); away_elo_pre.append(ra)
    exp_home.append(e_h);    hfa_used.append(hfa)

    s_h = 1.0 if row.FTR == "H" else (0.5 if row.FTR == "D" else 0.0)
    margin = abs(int(row.FTHG) - int(row.FTAG))
    k_eff = K * np.sqrt(max(margin, 1))              # margin-of-victory scaling
    elo[row.HomeTeam] = rh + k_eff * (s_h - e_h)
    elo[row.AwayTeam] = ra + k_eff * ((1 - s_h) - (1 - e_h))

matches["home_elo_pre"] = home_elo_pre
matches["away_elo_pre"] = away_elo_pre
matches["elo_diff"] = matches.home_elo_pre - matches.away_elo_pre
matches["elo_exp_home"] = exp_home
matches["hfa_used"] = hfa_used

_final = pd.Series(elo).sort_values(ascending=False)
print("\ncurrent Elo, top 8:")
print(_final.head(8).round(0).to_string())
print(f"\nMan United: {elo['Man United']:.0f}  (rank {_final.index.get_loc('Man United') + 1})")


### 3.4c Opponent-weighted form + derby flag

- **opponent-weighted form** — like rolling points, but each past result is scaled by the
  opponent's Elo at the time (beating a 1700 side counts more than beating a 1400 side).
  We attach the opponent's pre-match Elo to each `tm` row, then roll `pts * (opp_elo/1500)`.
- **derby flag** — hard-coded rival pairs; derbies suppress home advantage and raise cards.

In [ ]:
# opponent pre-match Elo onto team-match rows (from the match table, both sides)
_eh = matches[["match_id", "HomeTeam", "away_elo_pre"]].rename(
    columns={"HomeTeam": "team", "away_elo_pre": "opp_elo_pre"})
_ea = matches[["match_id", "AwayTeam", "home_elo_pre"]].rename(
    columns={"AwayTeam": "team", "home_elo_pre": "opp_elo_pre"})
opp_elo = pd.concat([_eh, _ea], ignore_index=True)
tm = tm.merge(opp_elo, on=["match_id", "team"], how="left")

tm["pts_x_oppstrength"] = tm.pts * (tm.opp_elo_pre / BASE)
tm["wform_roll5"] = prior_rolling(tm.pts_x_oppstrength, _key(tm.team), WINDOW)

DERBIES = {
    frozenset({"Arsenal", "Tottenham"}), frozenset({"Liverpool", "Everton"}),
    frozenset({"Man United", "Man City"}), frozenset({"Man United", "Liverpool"}),
    frozenset({"Chelsea", "Tottenham"}), frozenset({"Chelsea", "Arsenal"}),
    frozenset({"Chelsea", "Fulham"}), frozenset({"Arsenal", "Man United"}),
    frozenset({"Newcastle", "Sunderland"}), frozenset({"Aston Villa", "Birmingham"}),
    frozenset({"Aston Villa", "West Brom"}), frozenset({"Wolves", "West Brom"}),
    frozenset({"Southampton", "Portsmouth"}), frozenset({"Crystal Palace", "Brighton"}),
    frozenset({"West Ham", "Tottenham"}), frozenset({"West Ham", "Chelsea"}),
    frozenset({"West Ham", "Millwall"}), frozenset({"Nott'm Forest", "Derby"}),
    frozenset({"Leeds", "Man United"}),
}
matches["is_derby"] = [
    frozenset({h, a}) in DERBIES
    for h, a in zip(matches.HomeTeam, matches.AwayTeam)
]
print("derby matches:", matches.is_derby.sum(),
      "  H/D/A within derbies:",
      matches.loc[matches.is_derby, "FTR"].value_counts().to_dict())

### 3.4e  Market signal — vig-free probabilities & implied goal expectation

The bookmaker's price is the single best pre-match forecast in existence: it aggregates
every model, every injury leak and every stake. Beating *always-home* is a low bar;
**beating the market's log loss is the real test**. We turn the raw odds into features the
tree can use to learn only where public stats spot an inefficiency.

1. **De-vig the 1X2 price.** Raw implied probabilities `1/odds` sum to ~1.05 (the
   "overround"). `mkt_p_*` removes it **proportionally** (divide by the sum) — simple and,
   on this data, within a whisker of optimal. `mkt_pow_*` is a second view that also
   corrects the favourite–longshot bias (`p ∝ (1/odds)^γ`, `γ` fit once on 2000–2018).
2. **Implied total goals.** De-vig the Over/Under 2.5 line, then invert the Poisson tail
   `P(N ≥ 3) = 1 − e^{−λ}(1 + λ + λ²/2)` for the total-goals mean `λ_tot`. Split it with the
   1X2 supremacy → `mkt_xg_home`, `mkt_xg_away` (a clean prior for Dixon-Coles).
3. **Closing-line value.** `clv_* = close_p − open_p` — how far the market moved after the
   open, the sharp-money direction.

All of it is strictly pre-kickoff.


In [ ]:
def devig_proportional(odds_df):
    inv = 1.0 / odds_df.to_numpy(float)
    return inv / inv.sum(axis=1, keepdims=True)


def fit_power_gamma(odds_df, y_idx, grid=np.linspace(0.80, 1.10, 61)):
    """Favourite-longshot-corrected de-vig: p_i propto (1/o_i)^gamma. Pick gamma that
    minimises 1X2 log loss on the rows given (y_idx = 0/1/2)."""
    inv = 1.0 / odds_df.to_numpy(float)
    best, best_ll = 1.0, np.inf
    for g in grid:
        p = inv ** g
        p = p / p.sum(axis=1, keepdims=True)
        ll = -np.log(np.clip(p[np.arange(len(p)), y_idx], 1e-12, 1)).mean()
        if ll < best_ll:
            best, best_ll = float(g), ll
    return best


def total_goals_from_over(p_over, lo=0.15, hi=7.0, iters=70):
    """Invert P(N>=3) = 1 - e^-L (1 + L + L^2/2) for the Poisson total-goals mean L."""
    p_over = np.asarray(p_over, float)
    lo = np.full_like(p_over, lo); hi = np.full_like(p_over, hi)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        p_mid = 1.0 - np.exp(-mid) * (1 + mid + mid**2 / 2)
        lo = np.where(p_mid < p_over, mid, lo)
        hi = np.where(p_mid < p_over, hi, mid)
    return 0.5 * (lo + hi)


# --- 1X2 de-vig (open + close) --------------------------------------------------
o = matches[["mkt_H", "mkt_D", "mkt_A"]]
ok = o.notna().all(axis=1)
y_idx_all = matches.FTR.map({"H": 0, "D": 1, "A": 2}).to_numpy()

for col in ["mkt_p_H", "mkt_p_D", "mkt_p_A", "mkt_pow_H", "mkt_pow_D", "mkt_pow_A"]:
    matches[col] = np.nan
matches.loc[ok, ["mkt_p_H", "mkt_p_D", "mkt_p_A"]] = devig_proportional(o[ok])

fit_mask = ok & (matches.Season <= "2018-19")
GAMMA = fit_power_gamma(o[fit_mask], y_idx_all[fit_mask.to_numpy()])
pw = (1.0 / o[ok].to_numpy(float)) ** GAMMA
matches.loc[ok, ["mkt_pow_H", "mkt_pow_D", "mkt_pow_A"]] = pw / pw.sum(axis=1, keepdims=True)

matches["mkt_overround"] = (1.0 / o).sum(axis=1) - 1.0
matches["mkt_supremacy"] = matches.mkt_p_H - matches.mkt_p_A
_p3 = matches[["mkt_p_H", "mkt_p_D", "mkt_p_A"]].clip(1e-9)
matches["mkt_entropy"] = -(_p3 * np.log(_p3)).sum(axis=1)

c = matches[["close_H", "close_D", "close_A"]]
ck = c.notna().all(axis=1)
close_p = pd.DataFrame(np.nan, index=matches.index, columns=["close_p_H", "close_p_D", "close_p_A"])
close_p.loc[ck, :] = devig_proportional(c[ck])
matches[["close_p_H", "close_p_D", "close_p_A"]] = close_p
for s in ["H", "D", "A"]:
    matches[f"clv_{s}"] = matches[f"close_p_{s}"] - matches[f"mkt_p_{s}"]

# --- implied goal expectation from Over/Under 2.5 -------------------------------
ou = matches[["mkt_over25", "mkt_under25"]]
ouk = ou.notna().all(axis=1)
p_over = pd.Series(np.nan, index=matches.index)
p_over.loc[ouk] = devig_proportional(ou[ouk])[:, 0]
matches["mkt_p_over25"] = p_over
matches["mkt_tot_goals"] = total_goals_from_over(p_over.fillna(0.5))
matches.loc[~ouk, "mkt_tot_goals"] = np.nan
share_home = (0.5 + 0.9 * matches.mkt_supremacy).clip(0.15, 0.85)
matches["mkt_xg_home"] = matches.mkt_tot_goals * share_home
matches["mkt_xg_away"] = matches.mkt_tot_goals * (1 - share_home)

MARKET_MATCH_FEATS = [
    "mkt_p_H", "mkt_p_D", "mkt_p_A", "mkt_pow_H", "mkt_pow_D", "mkt_pow_A",
    "mkt_overround", "mkt_supremacy", "mkt_entropy", "mkt_p_over25",
    "mkt_tot_goals", "mkt_xg_home", "mkt_xg_away",
    "close_p_H", "close_p_D", "close_p_A", "clv_H", "clv_D", "clv_A",
]
cov = float(matches["mkt_p_H"].notna().mean())
print(f"market features built — 1X2 de-vig coverage {cov:.1%}, "
      f"O/U coverage {ouk.mean():.1%}, closing coverage {ck.mean():.1%}, power gamma {GAMMA:.3f}")
print("mean de-vigged P(H/D/A):",
      matches.loc[ok, ["mkt_p_H", "mkt_p_D", "mkt_p_A"]].mean().round(4).to_dict(),
      " actual:", matches.FTR.value_counts(normalize=True).round(4).to_dict())
matches.loc[matches.Season == "2024-25",
           ["HomeTeam", "AwayTeam", "mkt_p_H", "mkt_p_D", "mkt_p_A",
            "mkt_tot_goals", "mkt_xg_home", "mkt_xg_away"]].head(6)


### 3.4d League position & relegation pressure

Running league table **as it stood before each match** (points, goal difference, rank).
Then two pressure features, more meaningful late in the season:

- `pts_gap_to_safety` — points above/below 18th place
- `pts_gap_to_top4` — points behind 4th
- `late_season` — match round > 28, when relegation/European battles intensify

Each requires the table *before* the current match, so we sort by date and compute the
standing incrementally. Round number ~ ceil(games_played_by_league / 10).

In [ ]:
# build the pre-match standing from tm (one row per team per match, chronological)
tm = tm.sort_values(["Season", "Date", "match_id"]).reset_index(drop=True)


def prematch_standing(g):
    """Within one season's rows: cumulative points/GD/rank BEFORE each team-match."""
    g = g.sort_values(["Date", "match_id"]).copy()
    g["cum_pts"] = g.groupby("team")["pts"].cumsum() - g["pts"]
    g["cum_gd"] = g.groupby("team")["gd"].cumsum() - g["gd"]
    g["rank"] = np.nan
    g["pts_of_18th"] = np.nan
    g["pts_of_4th"] = np.nan
    for dt in g["Date"].unique():
        seen = g[g["Date"] <= dt].sort_values("match_id")
        latest = seen.groupby("team")[["cum_pts", "cum_gd"]].last()
        order = latest.sort_values(["cum_pts", "cum_gd"], ascending=False)
        rk = {t: i + 1 for i, t in enumerate(order.index)}
        sel = g["Date"] == dt
        g.loc[sel, "rank"] = g.loc[sel, "team"].map(rk)
        if len(order) >= 18:
            g.loc[sel, "pts_of_18th"] = order["cum_pts"].iloc[17]
        if len(order) >= 4:
            g.loc[sel, "pts_of_4th"] = order["cum_pts"].iloc[3]
    return g


tm = pd.concat([prematch_standing(g) for _, g in tm.groupby("Season", sort=False)],
               ignore_index=True)
tm["pts_gap_to_safety"] = tm.cum_pts - tm.pts_of_18th
tm["pts_gap_to_top4"] = tm.cum_pts - tm.pts_of_4th
tm["match_round"] = tm.games_played.clip(upper=38)
tm["late_season"] = (tm.match_round >= 28).astype(int)

(tm[(tm.team == EXAMPLE_TEAM) & (tm.Season == "2015-16")]
   .filter(regex="team|Season|Date|rank|cum_pts|pts_gap|late_season")
   .tail(8))


## 3.5 Merge features back onto the match table

Split `tm` into home rows / away rows, prefix `home_` / `away_`, join on `match_id`.
Then add `*_diff` (home − away) columns.

In [ ]:
EWM_STATS = [f"{s}_ewm" for s in ROLL_STATS]        # pts_ewm, gf_ewm, ...
XG_FEATURES = ["xg_roll5", "xga_roll5", "xg_ewm", "xga_ewm",
               "xgd_roll5", "xgd_ewm", "finishing_roll5", "def_luck_roll5"]

TEAM_FEATURES = [
    f"pts_roll{WINDOW}", f"gf_roll{WINDOW}", f"ga_roll{WINDOW}", f"gd_roll{WINDOW}",
    f"shots_roll{WINDOW}", f"sot_roll{WINDOW}", f"corners_roll{WINDOW}",
    *EWM_STATS,
    "pts_std", "gf_std", "ga_std", "gd_std", "shots_std", "sot_std", "corners_std",
    f"venue_pts_roll{WINDOW}", "days_rest", "congestion_14d", "h2h_pts_roll5",
    "conv_roll5", "sot_rate_roll5", "save_pct_roll5", "wform_roll5",
    *XG_FEATURES,
    "cum_pts", "cum_gd", "rank", "pts_gap_to_safety", "pts_gap_to_top4",
    "match_round", "games_played",
]

home_feats = (tm[tm.venue == "home"].set_index("match_id")[TEAM_FEATURES]
              .add_prefix("home_"))
away_feats = (tm[tm.venue == "away"].set_index("match_id")[TEAM_FEATURES]
              .add_prefix("away_"))

model_df = (matches.set_index("match_id")
            .join(home_feats).join(away_feats)
            .reset_index())

for col in TEAM_FEATURES:
    model_df[f"{col}_diff"] = model_df[f"home_{col}"] - model_df[f"away_{col}"]

# match-level features already on `matches`: elo_diff, elo_exp_home, hfa_used,
# home/away_elo_pre, is_derby, and every MARKET_MATCH_FEATS column.
model_df["late_season"] = model_df["home_match_round"].ge(28).astype(int)
# goal-model targets (post-match, used only as y in model.ipynb, never as X)
model_df["home_goals"] = model_df["FTHG"]
model_df["away_goals"] = model_df["FTAG"]

model_df["target"] = model_df["FTR"]
print(model_df.shape)
print(f"{sum(c.endswith('_diff') for c in model_df.columns)} *_diff columns, "
      f"{len(MARKET_MATCH_FEATS)} market match-level features")
[c for c in model_df.columns if c.endswith("_diff")]


## 3.6 Leakage checks + train-ready view

1. **Independent recompute** of one rolling feature must match exactly.
2. **Elo must be strictly pre-match** — the first match of the dataset has both teams at
   exactly 1500.
3. Drop cold-start rows (either team < 3 games this season) into `train_ready`.
4. Save `data/model_df.parquet`.

In [ ]:
# 1. recompute home_gf_std independently, for a sample of matches
def naive_prior_gf_mean(team, season, upto):
    m = matches[(matches.Season == season) & (matches.Date < upto)
                & ((matches.HomeTeam == team) | (matches.AwayTeam == team))]
    gf = pd.concat([m.loc[m.HomeTeam == team, "FTHG"],
                    m.loc[m.AwayTeam == team, "FTAG"]])
    return gf.mean()

chk = model_df[model_df.home_games_played >= 3].sample(150, random_state=0)
err = max(abs(naive_prior_gf_mean(r.HomeTeam, r.Season, r.Date) - r.home_gf_std)
          for r in chk.itertuples(index=False))
print(f"max |home_gf_std - independent recompute|: {err:.2e}   (want ~0)")

# 1b. same check, spelled out for one Man United home match
mu = model_df[(model_df.HomeTeam == "Man United") & (model_df.Season == "2015-16")
              & (model_df.home_games_played >= 3)].iloc[0]
print(f"\nMan United home vs {mu.AwayTeam} on {mu.Date.date()} "
      f"(game {int(mu.home_games_played) + 1} of the season):")
print(f"  feature home_gf_std      = {mu.home_gf_std:.4f}")
print(f"  independent recompute    = {naive_prior_gf_mean('Man United', '2015-16', mu.Date):.4f}")

# 2. first match: both Elo == 1500
first = model_df.iloc[0]
print(f"\nfirst match Elo: home={first.home_elo_pre:.1f} away={first.away_elo_pre:.1f}  (want 1500 / 1500)")

# 3. train_ready
train_ready = model_df[(model_df.home_games_played >= 3)
                       & (model_df.away_games_played >= 3)].copy()
print(f"\nmodel_df {len(model_df)}  ->  train_ready {len(train_ready)}")
print((train_ready.target.value_counts(normalize=True) * 100).round(1).to_string())
print("\n(model_df.parquet is written at the end of 3.7, after the squad merge)")


## 3.7 Squad-strength module (FPL Core Insights)

Source: **[FPL-Core-Insights](https://github.com/olbauday/FPL-Core-Insights)** — per-season
`players.csv` (squad rosters), `playerstats.csv` (per-gameweek FPL stats), `teams.csv`.
Files in `data/fpl/<season>/`.

FPL player **price** (`now_cost`, in £m) is set before each season from prior-season output
and transfer activity — a clean pre-kickoff proxy for player quality. Per `(Season, team)`:

| feature | how |
|---|---|
| `squad_price_total` | sum of every registered player's pre-season price |
| `squad_price_top11` | price of the 11 most expensive players — the likely first XI |
| `bench_price` | price of squad players ranked 12–20 — depth |
| `gk_price` / `def_price` / `mid_price` / `fwd_price` | best GK, mean of top-5 def, top-5 mid, top-3 fwd — line strength |
| `squad_ppg` | mean prior-season points-per-game across the first XI |

Then merged as `home_* / away_* / *_diff`, plus **`att_edge_diff`** (my forwards vs your
defenders, minus the reverse) — the FM-style line match-up.

### Coverage & leakage

FPL data begins at **2024-25**, so squad features exist for **2024-25, 2025-26, 2026-27**
only; earlier seasons are `NaN` (tree models tolerate it; or train on the covered slice).

Leakage rule: season *S* uses the **GW1 snapshot** of season *S* — prices and
prior-season PPG are fixed before a ball is kicked. We do **not** touch mid-season price
changes, form, or xG. The **2026-27** rosters reflect the completed summer transfer window.


In [ ]:
FPL_DIR = os.path.join(DATA_DIR, "fpl")


def fpl_season_label(folder):
    """FPL season folder '2025-2026' -> our label '2025-26'."""
    a, b = folder.split("-")
    return f"{a}-{b[-2:]}"


# FPL team name -> football-data name  (names vary a little year to year)
FPL_TEAM_MAP = {
    "Arsenal": "Arsenal", "Aston Villa": "Aston Villa", "Bournemouth": "Bournemouth",
    "Brentford": "Brentford", "Brighton": "Brighton", "Burnley": "Burnley",
    "Chelsea": "Chelsea", "Coventry City": "Coventry", "Crystal Palace": "Crystal Palace",
    "Everton": "Everton", "Fulham": "Fulham", "Hull City": "Hull",
    "Ipswich": "Ipswich", "Ipswich Town": "Ipswich",
    "Leeds": "Leeds", "Leicester": "Leicester", "Liverpool": "Liverpool",
    "Man City": "Man City", "Man Utd": "Man United", "Newcastle": "Newcastle",
    "Nott'm Forest": "Nott'm Forest", "Southampton": "Southampton", "Spurs": "Tottenham",
    "Sunderland": "Sunderland", "West Ham": "West Ham", "Wolves": "Wolves",
}

POSITION_LINE = {"Goalkeeper": "gk", "Defender": "def", "Midfielder": "mid", "Forward": "fwd"}
FPL_STAT_COLS = ["id", "now_cost", "points_per_game"]   # pre-season-safe signals


def load_fpl_season(folder):
    """Load one FPL season: roster + GW1 snapshot, joined and mapped to our team names.

    `now_cost` in this dataset is already in £m (e.g. 4.0 .. 15.5); no rescaling.
    """
    base = os.path.join(FPL_DIR, folder)
    players = pd.read_csv(os.path.join(base, "players.csv"))
    teams = pd.read_csv(os.path.join(base, "teams.csv"))
    stats = pd.read_csv(os.path.join(base, "playerstats.csv"), low_memory=False)

    gw1 = stats[stats["gw"] == stats["gw"].min()][FPL_STAT_COLS].copy()

    df = players.merge(gw1, left_on="player_id", right_on="id", how="left")
    df = df.merge(teams[["code", "name"]], left_on="team_code", right_on="code", how="left")
    df["team"] = df["name"].map(FPL_TEAM_MAP)
    df["Season"] = fpl_season_label(folder)
    df["line"] = df["position"].map(POSITION_LINE)
    df["price"] = pd.to_numeric(df["now_cost"], errors="coerce")          # £m
    df["ppg"] = pd.to_numeric(df["points_per_game"], errors="coerce")

    unmapped = sorted(set(df.loc[df["team"].isna(), "name"].dropna()))
    if unmapped:
        print(f"  ! {folder}: unmapped clubs {unmapped}")
    return df[df["team"].notna()].copy()


fpl_folders = sorted(d for d in os.listdir(FPL_DIR)
                     if os.path.isdir(os.path.join(FPL_DIR, d)))
print(f"{len(fpl_folders)} FPL season folder(s):")
for f in fpl_folders:
    d = load_fpl_season(f)
    print(f"  {f}  ->  season {fpl_season_label(f)}   "
          f"({len(d)} PL players, {d.team.nunique()} clubs)")


In [ ]:
LINE_TOP_N = {"def": 5, "mid": 5, "fwd": 3}   # players per line that define its strength


def aggregate_fpl_season(df):
    """One FPL season (roster + GW1 snapshot) -> one row per (Season, team). Prices in £m."""
    rows = []
    for (season, team), g in df.groupby(["Season", "team"]):
        g = g.sort_values("price", ascending=False)
        top11 = g.head(11)
        bench = g.iloc[11:20]
        rec = {
            "Season": season, "team": team,
            "squad_size": len(g),
            "squad_price_total": g.price.sum(min_count=1),
            "squad_price_top11": top11.price.sum(min_count=1),
            "bench_price": bench.price.sum(min_count=1) if len(bench) else np.nan,
            "squad_ppg": top11.ppg.mean(),
            "gk_price": g.loc[g.line == "gk", "price"].max(),
        }
        for line, n in LINE_TOP_N.items():
            rec[f"{line}_price"] = g.loc[g.line == line, "price"].head(n).mean()
        rows.append(rec)
    return pd.DataFrame(rows)


strength = pd.concat([aggregate_fpl_season(load_fpl_season(f)) for f in fpl_folders],
                     ignore_index=True)
strength.to_csv(os.path.join(DATA_DIR, "team_season_strength.csv"), index=False)

print(strength.shape, "team-seasons  |  seasons:", sorted(strength.Season.unique()))
print("\n2026-27 (post-transfer-window squads), by first-XI price:")
print(strength[strength.Season == "2026-27"]
      .sort_values("squad_price_top11", ascending=False)
      [["team", "squad_price_top11", "fwd_price", "mid_price", "def_price", "gk_price"]]
      .head(10).to_string(index=False))


In [ ]:
SQUAD_FEATURES = [
    "squad_price_total", "squad_price_top11", "bench_price", "squad_ppg",
    "gk_price", "def_price", "mid_price", "fwd_price",
]

h = (strength.set_index(["Season", "team"])[SQUAD_FEATURES]
     .add_prefix("home_").rename_axis(index={"team": "HomeTeam"}))
a = (strength.set_index(["Season", "team"])[SQUAD_FEATURES]
     .add_prefix("away_").rename_axis(index={"team": "AwayTeam"}))

model_df = (model_df.merge(h.reset_index(), on=["Season", "HomeTeam"], how="left")
                    .merge(a.reset_index(), on=["Season", "AwayTeam"], how="left"))

for f in SQUAD_FEATURES:
    model_df[f"{f}_diff"] = model_df[f"home_{f}"] - model_df[f"away_{f}"]

# FM-style line match-ups: my attack vs your defence
model_df["home_fwd_vs_away_def"] = model_df.home_fwd_price - model_df.away_def_price
model_df["away_fwd_vs_home_def"] = model_df.away_fwd_price - model_df.home_def_price
model_df["att_edge_diff"] = model_df.home_fwd_vs_away_def - model_df.away_fwd_vs_home_def

covered = model_df.home_squad_price_top11.notna().sum()
print(f"squad features merged; {covered}/{len(model_df)} matches covered "
      f"({covered / len(model_df):.0%})")
print("seasons covered:",
      sorted(model_df.loc[model_df.home_squad_price_top11.notna(), "Season"].unique()))

model_df.to_parquet(os.path.join(DATA_DIR, "model_df.parquet"), index=False)
print(f"\nwrote data/model_df.parquet  {model_df.shape}")
